<a href="https://colab.research.google.com/github/mjgpinheiro/Physics_models/blob/main/ratio_estimator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
ratio_estimator.py — release version
====================================
Reproduction script for:
"Directional Cross-Scale Operators from Lagged Covariance Ratios:
 Identifiability and Consistency under Measurement Noise"

Single self-contained module. No self-import.

Experiments (defaults use tau=1; the adaptive moment rule appears
only in E4, where it is evaluated and found not to improve on tau=1):
  E1: noiseless baseline, ratio (several taus) vs YW/OLS
  E2: measurement-noise sweep at fixed T, with IQRs, tau=1
  E3: err(T) across four SNRs vs predicted YW plateau, tau=1
  E4: lag-selection: moment rule vs oracle vs fixed tau=1
  E5: asymptotic rate at 0 dB, tau=1, with per-realisation
      beta_1 slopes and their CI
  E6: non-proportional (scale-confined) noise: shape distortion of
      YW vs closed-form limit (Prop. 4); ratio remains consistent

Measurement-noise model (proportional, as in Prop. 3 special case):
  Sigma_eta = nu * Sigma,   Sigma = population stationary covariance
  (discrete Lyapunov equation),  eta_t = L z_t,  L = chol(nu * Sigma).
  E6 adds a non-proportional model: noise confined to the fine half
  of the scales, matched in trace to the same nominal SNR.

All randomness is seeded. Ground truth is used only in evaluation.
"""

import numpy as np
from scipy.linalg import solve_discrete_lyapunov

# ────────────────────────────────────────────────────────────────
# System
# ────────────────────────────────────────────────────────────────
def make_system(n=12, a=0.92, g=0.15, alpha=0.6):
    idx = np.arange(n)
    K = alpha * np.exp(-np.abs(idx[:, None] - idx[None, :]) / 2.0) \
              * np.sign(idx[None, :] - idx[:, None])
    A = a * np.eye(n) + g * K
    G = g * K
    Gas = 0.5 * (G - G.T)
    return A, G, Gas

def stationary_cov(A, sigma_eps=0.20):
    """Population stationary covariance from the discrete Lyapunov
    equation  Sigma = A Sigma A^T + sigma_eps^2 I  (exact, trajectory-
    independent)."""
    n = A.shape[0]
    return solve_discrete_lyapunov(A, sigma_eps**2 * np.eye(n))

def simulate(A, T, sigma_eps=0.20, seed=0, burn=500):
    rng = np.random.default_rng(seed)
    n = A.shape[0]
    W = np.zeros(n)
    out = np.empty((T, n))
    for t in range(burn + T):
        W = A @ W + sigma_eps * rng.standard_normal(n)
        if t >= burn:
            out[t - burn] = W
    return out

def add_proportional_noise(W, snr_db, Sigma, seed=0):
    """Y = W + eta with Sigma_eta = nu * Sigma (proportional model),
    where Sigma is the POPULATION stationary covariance from the
    Lyapunov equation. L is trajectory-independent, so eta is genuinely
    independent of W and Sigma_eta = nu*Sigma holds exactly."""
    rng = np.random.default_rng(seed + 777)
    nu = 10 ** (-snr_db / 10.0)
    L = np.linalg.cholesky(nu * Sigma)
    eta = rng.standard_normal(W.shape) @ L.T
    return W + eta, nu

# ────────────────────────────────────────────────────────────────
# Estimators
# ────────────────────────────────────────────────────────────────
def sample_cov(Y, tau):
    T = Y.shape[0]
    return Y[:T - tau].T @ Y[tau:] / (T - tau)

def estimate_ratio(Y, tau=1):
    """A^T = R(tau)^{-1} R(tau+1). Default tau=1 (paper default)."""
    Rt  = sample_cov(Y, tau)
    Rt1 = sample_cov(Y, tau + 1)
    return np.linalg.solve(Rt, Rt1).T

def estimate_yw(Y):
    """Yule-Walker: A^T = R(0)^{-1} R(1). Same plim as OLS."""
    return np.linalg.solve(sample_cov(Y, 0), sample_cov(Y, 1)).T

# ────────────────────────────────────────────────────────────────
# Metrics (primary: operator/amplitude; corr secondary)
# ────────────────────────────────────────────────────────────────
def metrics(A_hat, A, Gas):
    Gas_hat = 0.5 * (A_hat - A_hat.T)
    return dict(
        errA   = np.linalg.norm(A_hat - A) / np.linalg.norm(A),
        errGas = np.linalg.norm(Gas_hat - Gas) / np.linalg.norm(Gas),
        b_amp  = np.linalg.norm(Gas_hat) / np.linalg.norm(Gas),
        corr   = np.corrcoef(Gas_hat.ravel(), Gas.ravel())[0, 1],
    )

def med_iqr(vals):
    v = np.asarray(vals)
    return (np.median(v), np.percentile(v, 25), np.percentile(v, 75))

# ────────────────────────────────────────────────────────────────
# Adaptive lag selection (E4 only; not used in main experiments)
# ────────────────────────────────────────────────────────────────
def select_tau_moment(Y, tau_max=10):
    T = Y.shape[0]
    Y_est, Y_val = Y[:T // 2], Y[T // 2:]
    best_tau, best_res = 1, np.inf
    for tau in range(1, tau_max + 1):
        try:
            A_hat = estimate_ratio(Y_est, tau)
        except np.linalg.LinAlgError:
            continue
        Rv_t  = sample_cov(Y_val, tau)
        Rv_t1 = sample_cov(Y_val, tau + 1)
        res = np.linalg.norm(Rv_t1 - Rv_t @ A_hat.T) \
            / np.linalg.norm(Rv_t1)
        if res < best_res:
            best_res, best_tau = res, tau
    return best_tau

# ────────────────────────────────────────────────────────────────
# Experiments
# ────────────────────────────────────────────────────────────────
def E1(T=10_000, seed=42):
    A, G, Gas = make_system()
    W = simulate(A, T, seed=seed)
    print(f"rho(A) = {np.max(np.abs(np.linalg.eigvals(A))):.7f}")
    Sig = W.T @ W / T
    ev = np.linalg.eigvalsh(Sig)
    print(f"kappa(Sigma_hat) = {ev[-1]/ev[0]:.4f}")
    print("\nE1 — noiseless baseline")
    print(f"{'method':<10}{'tau':>4}{'errGas':>9}{'b_amp':>8}{'corr':>9}")
    m = metrics(estimate_yw(W), A, Gas)
    print(f"{'YW/OLS':<10}{'-':>4}{m['errGas']:>9.3f}{m['b_amp']:>8.3f}"
          f"{m['corr']:>9.4f}")
    for tau in (1, 2, 5, 10):
        m = metrics(estimate_ratio(W, tau), A, Gas)
        print(f"{'ratio':<10}{tau:>4}{m['errGas']:>9.3f}{m['b_amp']:>8.3f}"
              f"{m['corr']:>9.4f}")

def E2(T=10_000, n_real=50, snrs=(30, 20, 10, 5, 0)):
    A, G, Gas = make_system()
    Sigma = stationary_cov(A)
    print(f"\nE2 — proportional-noise sweep (T={T}, {n_real} realisations,"
          " tau=1, population Sigma)")
    hdr = (f"{'SNR':>4} | errGas_r med[IQR]     | errGas_YW med[IQR]    |"
           f" b_amp_r med[IQR]      | b_amp_YW med[IQR]     | 1/(1+nu)"
           f" | corr_r  | corr_YW")
    print(hdr)
    for snr in snrs:
        rr, ry = [], []
        for k in range(n_real):
            W = simulate(A, T, seed=1000 + k)
            Y, nu = add_proportional_noise(W, snr, Sigma, seed=k)
            rr.append(metrics(estimate_ratio(Y, 1), A, Gas))
            ry.append(metrics(estimate_yw(Y), A, Gas))
        def s(rs, key):
            m, lo, hi = med_iqr([r[key] for r in rs])
            return f"{m:.3f} [{lo:.3f},{hi:.3f}]"
        cr = np.median([r['corr'] for r in rr])
        cy = np.median([r['corr'] for r in ry])
        print(f"{snr:>4} | {s(rr,'errGas'):>21} | {s(ry,'errGas'):>21} |"
              f" {s(rr,'b_amp'):>21} | {s(ry,'b_amp'):>21} |"
              f" {1/(1+nu):>8.3f} | {cr:.4f} | {cy:.4f}")

def E3(snrs=(20, 10, 5, 0), Ts=(10_000, 30_000, 100_000), n_real=10):
    A, G, Gas = make_system()
    Sigma = stationary_cov(A)
    print(f"\nE3 — errA(T) per SNR (medians over {n_real} realisations,"
          " tau=1)")
    print(f"{'SNR':>4}{'T':>9}{'errA_ratio':>12}{'errA_YW':>10}"
          f"{'plateau':>9}")
    for T in Ts:
        Ws = [simulate(A, T, seed=3000 + k) for k in range(n_real)]
        for snr in snrs:
            er, ey = [], []
            for k, W in enumerate(Ws):
                Y, nu = add_proportional_noise(W, snr, Sigma, seed=k)
                er.append(metrics(estimate_ratio(Y, 1), A, Gas)['errA'])
                ey.append(metrics(estimate_yw(Y), A, Gas)['errA'])
            print(f"{snr:>4}{T:>9}{np.median(er):>12.4f}"
                  f"{np.median(ey):>10.4f}{nu/(1+nu):>9.4f}")

def E4(snr=10, T=10_000, n_real=50, tau_max=10):
    A, G, Gas = make_system()
    Sigma = stationary_cov(A)
    print(f"\nE4 — lag selection: moment rule vs oracle vs tau=1 "
          f"(SNR={snr} dB, T={T}, {n_real} realisations)")
    reg_m, reg_1, t_sel, t_ora = [], [], [], []
    for k in range(n_real):
        W = simulate(A, T, seed=4000 + k)
        Y, _ = add_proportional_noise(W, snr, Sigma, seed=k)
        errs = {}
        for tau in range(1, tau_max + 1):
            try:
                errs[tau] = metrics(estimate_ratio(Y, tau), A, Gas)['errGas']
            except np.linalg.LinAlgError:
                pass
        tau_o = min(errs, key=errs.get)
        tau_s = select_tau_moment(Y, tau_max)
        reg_m.append((errs[tau_s] - errs[tau_o]) / errs[tau_o])
        reg_1.append((errs[1]     - errs[tau_o]) / errs[tau_o])
        t_sel.append(tau_s); t_ora.append(tau_o)
    for name, reg in (("moment rule", reg_m), ("fixed tau=1", reg_1)):
        m, lo, hi = med_iqr(reg)
        p90 = np.percentile(reg, 90)
        frac = np.mean(np.array(reg) > 0)
        print(f"  {name:<12}: median {m:.4f} [IQR {lo:.4f},{hi:.4f}]"
              f"  P90 {p90:.4f}  max {max(reg):.4f}"
              f"  regret>0 in {100*frac:.0f}%")
    from collections import Counter
    print(f"  selected taus: {dict(Counter(t_sel))}")
    print(f"  oracle   taus: {dict(Counter(t_ora))}")

def E5(snr_db=0, Ts=(10_000, 30_000, 100_000, 300_000), n_real=10):
    A, G, Gas = make_system()
    Sigma = stationary_cov(A)
    print(f"\nE5 — asymptotic rate at {snr_db} dB "
          f"({n_real} realisations, tau=1)")
    print(f"{'T':>8}{'errA_ratio':>12}{'errA_YW':>10}"
          f"{'b_amp_r':>9}{'b_amp_YW':>10}")
    errs = np.empty((n_real, len(Ts)))          # for beta_1 slopes
    for j, T in enumerate(Ts):
        er, ey, br, by = [], [], [], []
        for k in range(n_real):
            W = simulate(A, T, seed=5000 + k)
            Y, nu = add_proportional_noise(W, snr_db, Sigma, seed=k)
            mr = metrics(estimate_ratio(Y, 1), A, Gas)
            my = metrics(estimate_yw(Y), A, Gas)
            er.append(mr['errA']); ey.append(my['errA'])
            br.append(mr['b_amp']); by.append(my['b_amp'])
            errs[k, j] = mr['errA']
        print(f"{T:>8}{np.median(er):>12.4f}{np.median(ey):>10.4f}"
              f"{np.median(br):>9.3f}{np.median(by):>10.3f}")
    print(f"  predicted YW plateau nu/(1+nu) = "
          f"{(10**(-snr_db/10))/(1+10**(-snr_db/10)):.3f}")
    # beta_1 from per-realisation slopes (independent units)
    logT = np.log(np.array(Ts))
    slopes = []
    for k in range(n_real):
        b = np.polyfit(logT, np.log(errs[k]), 1)[0]
        slopes.append(b)
    slopes = np.array(slopes)
    mean = slopes.mean()
    se = slopes.std(ddof=1) / np.sqrt(n_real)
    # t-based 95% CI, df = n_real-1 (t_{0.975,9} = 2.262)
    tcrit = 2.262
    print(f"  per-realisation slopes: "
          + ", ".join(f"{s:.4f}" for s in slopes))
    print(f"  beta_1 = {mean:.4f} +/- {tcrit*se:.4f} "
          f"(95% CI from {n_real} independent slopes; prediction -0.5)")


# ────────────────────────────────────────────────────────────────
# E6: non-proportional (scale-confined) measurement noise
# ────────────────────────────────────────────────────────────────
def add_confined_noise(W, snr_db, Sigma, seed=0, mask=None):
    """Y = W + eta with Sigma_eta = diag(d), d_i = 0 outside `mask`
    and constant inside, scaled so that tr Sigma_eta = nu * tr Sigma
    (same nominal SNR as the proportional model). Default mask: the
    fine half of the scales, i >= n/2. eta is independent of W."""
    rng = np.random.default_rng(seed + 999)
    n = W.shape[1]
    if mask is None:
        mask = np.arange(n) >= n // 2
    nu = 10 ** (-snr_db / 10.0)
    d = mask.astype(float)
    d *= nu * np.trace(Sigma) / d.sum()
    eta = rng.standard_normal(W.shape) * np.sqrt(d)
    return W + eta, np.diag(d), nu

def yw_limit(A, Sigma, Sigma_eta):
    """Closed-form probability limit of YW/OLS under measurement noise
    (Prop. 4):  A_YW^T = (Sigma + Sigma_eta)^{-1} Sigma A^T."""
    return np.linalg.solve(Sigma + Sigma_eta, Sigma @ A.T).T

def E6(snrs=(5, 0), Ts=(10_000, 30_000, 100_000), n_real=10):
    A, G, Gas = make_system()
    Sigma = stationary_cov(A)
    n = A.shape[0]
    print(f"\nE6 — non-proportional noise confined to scales i >= {n//2}"
          f" (medians over {n_real} realisations, tau=1)")
    print(f"{'SNR':>4}{'T':>9}{'errGas_r':>10}{'errGas_YW':>11}"
          f"{'pred':>8}{'corr_r':>9}{'corr_YW':>9}{'pred':>8}"
          f"{'b_amp_YW':>10}{'pred':>8}")
    for snr in snrs:
        for T in Ts:
            er, ey, cr, cy, by = [], [], [], [], []
            for k in range(n_real):
                W = simulate(A, T, seed=6000 + k)
                Y, Se, nu = add_confined_noise(W, snr, Sigma, seed=k)
                mr = metrics(estimate_ratio(Y, 1), A, Gas)
                my = metrics(estimate_yw(Y), A, Gas)
                er.append(mr['errGas']); ey.append(my['errGas'])
                cr.append(mr['corr']);   cy.append(my['corr'])
                by.append(my['b_amp'])
            mp = metrics(yw_limit(A, Sigma, Se), A, Gas)   # closed form
            print(f"{snr:>4}{T:>9}{np.median(er):>10.4f}"
                  f"{np.median(ey):>11.4f}{mp['errGas']:>8.4f}"
                  f"{np.median(cr):>9.4f}{np.median(cy):>9.4f}"
                  f"{mp['corr']:>8.4f}{np.median(by):>10.3f}"
                  f"{mp['b_amp']:>8.3f}")

if __name__ == "__main__":
    E1()
    E2()
    E3()
    E4()
    E5()
    E6()


rho(A) = 0.9329583
kappa(Sigma_hat) = 1.4107

E1 — noiseless baseline
method     tau   errGas   b_amp     corr
YW/OLS       -    0.089   1.001   0.9960
ratio        1    0.091   1.011   0.9960
ratio        2    0.105   1.013   0.9947
ratio        5    0.135   1.006   0.9910
ratio       10    0.215   1.037   0.9784

E2 — proportional-noise sweep (T=10000, 50 realisations, tau=1, population Sigma)
 SNR | errGas_r med[IQR]     | errGas_YW med[IQR]    | b_amp_r med[IQR]      | b_amp_YW med[IQR]     | 1/(1+nu) | corr_r  | corr_YW
  30 |   0.109 [0.103,0.113] |   0.100 [0.095,0.105] |   1.009 [1.005,1.015] |   1.008 [1.003,1.015] |    0.999 | 0.9942 | 0.9951
  20 |   0.110 [0.104,0.117] |   0.101 [0.095,0.105] |   1.009 [1.003,1.015] |   1.001 [0.995,1.007] |    0.990 | 0.9941 | 0.9950
  10 |   0.124 [0.116,0.132] |   0.134 [0.130,0.142] |   1.014 [1.003,1.020] |   0.918 [0.912,0.925] |    0.909 | 0.9926 | 0.9935
   5 |   0.180 [0.170,0.194] |   0.266 [0.258,0.273] |   1.027 [1.012,1.037] | 